# Euclidean Embedding Dimension Optimization using BIC

This notebook determines the optimal embedding dimension for classical (Euclidean) Multidimensional Scaling (MDS) by calculating the Bayesian Information Criterion (BIC) for a range of dimensions. The dimension with the lowest BIC is considered optimal as it provides the best balance between model fit and complexity.

We will use functions from `analysis_from_mat.py` to perform the analysis.

## 1. Setup and Imports

First, we import the necessary libraries and functions. This includes `numpy` for numerical operations, `matplotlib` for plotting, and our custom functions for loading data and running the embeddings.

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# Ensure the script can find the analysis_from_mat module
# If this notebook is in the same directory as the .py file, '.' is sufficient.
if '.' not in sys.path:
    sys.path.append('.')

from analysis_from_mat import (
    load_dmat_from_mat, 
    run_euclidean_embedding, 
    plot_shepard_diagram
)

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')

## 2. Configuration and Data Loading

Specify the path to your `.mat` file, the name of the distance matrix variable within it, and the range of dimensions you want to test.

In [ ]:
# --- User Configuration ---
MAT_FILE_PATH = 'data/example_data.mat'  # <--- CHANGE THIS to the path of your .mat file
MATRIX_VARIABLE_NAME = 'dissimilarity_matrix' # <--- CHANGE THIS to the variable name in your file
UNC_VARIABLE_NAME = None  # Set to the name of your uncertainty matrix, or None if not applicable

# --- Analysis Parameters ---
DIM_RANGE = range(2, 11)  # Test dimensions from 2 to 10

# --- Load Data ---
try:
    dmat_original = load_dmat_from_mat(MAT_FILE_PATH, MATRIX_VARIABLE_NAME)
    dmat_unc = None
    if UNC_VARIABLE_NAME:
        print(f"Loading uncertainty data from variable: {UNC_VARIABLE_NAME}")
        dmat_unc = load_dmat_from_mat(MAT_FILE_PATH, UNC_VARIABLE_NAME)
    
    print(f"\nSuccessfully loaded distance matrix of shape {dmat_original.shape}")
except Exception as e:
    print(f"Error loading data: {e}")
    # Stop execution if data loading fails
    dmat_original = None

## 3. Run Euclidean MDS and Calculate BIC

We now loop through the specified dimension range. In each iteration, we run the Euclidean MDS embedding and store the resulting BIC value.

In [ ]:
bic_values = []
dimensions = list(DIM_RANGE)

if dmat_original is not None:
    for dim in dimensions:
        print(f"\n{'='*10} Running for Dimension: {dim} {'='*10}")
        
        # Run the Euclidean embedding
        # Note: The function handles normalization internally
        euclidean_results = run_euclidean_embedding(
            dmat=dmat_original.copy(), 
            embedding_dim=dim, 
            dmat_unc=dmat_unc.copy() if dmat_unc is not None else None
        )
        
        if euclidean_results:
            bic = euclidean_results['bic']
            bic_values.append(bic)
            print(f"Dimension {dim} -> BIC: {bic:.2f}")
        else:
            print(f"Failed to get results for dimension {dim}.")
            bic_values.append(np.nan)

    print("\nBIC calculation complete.")

## 4. Plot BIC vs. Dimension and Find Optimum

Visualizing the BIC values helps in identifying the 'elbow' or minimum point, which corresponds to the optimal dimension.

In [ ]:
if bic_values:
    # Find the optimal dimension (minimum BIC)
    min_bic_index = np.nanargmin(bic_values)
    optimal_dim = dimensions[min_bic_index]
    min_bic = bic_values[min_bic_index]

    print(f"Optimal Embedding Dimension: {optimal_dim} (BIC = {min_bic:.2f})")

    # Plot the results
    plt.figure(figsize=(10, 6))
    plt.plot(dimensions, bic_values, marker='o', linestyle='-', color='b', label='BIC')
    plt.scatter([optimal_dim], [min_bic], color='red', s=100, zorder=5, label=f'Optimal Dimension ({optimal_dim})')
    
    plt.title('BIC vs. Euclidean Embedding Dimension', fontsize=16)
    plt.xlabel('Embedding Dimension', fontsize=12)
    plt.ylabel('Bayesian Information Criterion (BIC)', fontsize=12)
    plt.xticks(dimensions)
    plt.legend()
    plt.show()
else:
    print("No BIC values were calculated. Cannot plot results.")
    optimal_dim = None

## 5. Analyze the Optimal Embedding

Now, we re-run the embedding with the optimal dimension and visualize the results, including a Shepard diagram and a PCA projection of the embedded points.

In [ ]:
if optimal_dim is not None:
    print(f"\n--- Analyzing Optimal Embedding for D = {optimal_dim} ---")
    
    # Re-run the embedding for the optimal dimension
    optimal_results = run_euclidean_embedding(
        dmat=dmat_original.copy(),
        embedding_dim=optimal_dim,
        dmat_unc=dmat_unc.copy() if dmat_unc is not None else None
    )

    # 1. Generate Shepard Diagram
    plot_shepard_diagram(
        original_dmat=optimal_results['dmat'],
        embedded_dmat=optimal_results['emb_mat'],
        title=f'Shepard Diagram: Optimal Euclidean Embedding (D={optimal_dim})',
        sample_fraction=0.1  # Sample 10% of points for a clearer plot if N is large
    )

    # 2. Visualize the Embedding (using PCA if D > 2)
    coords = optimal_results['coords']
    title_text = f'Euclidean Embedding (D={optimal_dim})'
    
    if optimal_dim > 2:
        print(f"\nProjecting {optimal_dim}D embedding to 2D using PCA for visualization.")
        pca = PCA(n_components=2)
        coords_2d = pca.fit_transform(coords)
        explained_variance = sum(pca.explained_variance_ratio_) * 100
        title_text = f'PCA Projection of {optimal_dim}D Euclidean Embedding\n(Explains {explained_variance:.2f}% of variance)'
    else:
        coords_2d = coords
        
    plt.figure(figsize=(8, 8))
    plt.scatter(coords_2d[:, 0], coords_2d[:, 1], alpha=0.7, edgecolors='k', linewidths=0.5)
    plt.title(title_text, fontsize=16)
    plt.xlabel('Principal Component 1' if optimal_dim > 2 else 'Dimension 1', fontsize=12)
    plt.ylabel('Principal Component 2' if optimal_dim > 2 else 'Dimension 2', fontsize=12)
    plt.gca().set_aspect('equal', adjustable='box')
    plt.show()